# RTL Prompt Repetition Experiments — Kaggle Notebook

Replication of arXiv:2512.14982 (prompt repetition for non-reasoning LLMs) across 7 RTL languages: Punjabi (Shahmukhi), Urdu, Pashto, Balochi, Arabic, Persian, Sindhi.

**Kaggle setup (do once):**
1. Settings → **Internet: ON** (needed to clone from GitHub)
2. Settings → **Accelerator: GPU T4x2** (free 30 GPU-hours/week)

**Why Kaggle:** files in `/kaggle/working` auto-persist with the notebook — no Drive mount needed, and a disconnect never loses results. Results are resume-safe: rerunning the same command continues from the last saved item.

In [ ]:
# CELL 1 — Output directory (auto-saves with the notebook)
import os
OUT = '/kaggle/working/rtl_results'
os.makedirs(OUT, exist_ok=True)
print('Results will be saved to:', OUT)

In [ ]:
# CELL 2 — Install dependencies
!pip install -q transformers torch pandas openpyxl scipy rapidfuzz sentencepiece

In [ ]:
# CELL 3 — Clone repository (requires Internet: ON)
import os
if not os.path.exists('/kaggle/working/prompt_repitition_research'):
    os.system('git clone https://github.com/FizzahNasir/prompt_repitition_research.git')

%cd /kaggle/working/prompt_repitition_research
os.system('git pull origin master')

In [ ]:
# CELL 4 — Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (experiments will be very slow)'}")

## Step 1 — Dry run (verifies prompts build correctly, no inference)

In [ ]:
# CELL 5 — Dry run: Arabic, all 4 methods
!python /kaggle/working/prompt_repitition_research/run_all_languages.py --dry-run --language ar --models qwen2.5-3b --methods baseline repetition cross_lingual_t1 cross_lingual_t2

## Step 2 — Run experiments (Arabic example, qwen2.5-3b)

One language per cell so a disconnect only affects that cell. All 4 methods: `baseline`, `repetition`, `cross_lingual_t1` (English bridge), `cross_lingual_t2` (native bridge).

In [ ]:
# CELL 6 — Arabic: baseline + repetition (~1-2 hrs on T4, 1,460 items)
!python /kaggle/working/prompt_repitition_research/run_all_languages.py --language ar --models qwen2.5-3b --methods baseline repetition --out {OUT}

In [ ]:
# CELL 7 — Arabic: cross-lingual methods
!python /kaggle/working/prompt_repitition_research/run_all_languages.py --language ar --models qwen2.5-3b --methods cross_lingual_t1 cross_lingual_t2 --out {OUT}

In [ ]:
# CELL 8 — Check saved results (also visible in the Output sidebar)
!ls -lh {OUT}

## Step 3 — Other languages

| Code | Language | Items |
|------|----------|-------|
| `ur` | Urdu | ~800 |
| `fa` | Persian | 2,884 |
| `sd` | Sindhi | 817 |
| `pa` | Punjabi (Shahmukhi) | ~800 |
| `ps` | Pashto | ~800 |
| `bal` | Balochi (5/7 tasks only) | 386 |

In [ ]:
# CELL 9 — Urdu (all 4 methods)
!python /kaggle/working/prompt_repitition_research/run_all_languages.py --language ur --models qwen2.5-3b --methods baseline repetition cross_lingual_t1 cross_lingual_t2 --out {OUT}

In [ ]:
# CELL 10 — Persian (largest dataset: 2,884 items)
!python /kaggle/working/prompt_repitition_research/run_all_languages.py --language fa --models qwen2.5-3b --methods baseline repetition cross_lingual_t1 cross_lingual_t2 --out {OUT}

In [ ]:
# CELL 11 — Sindhi
!python /kaggle/working/prompt_repitition_research/run_all_languages.py --language sd --models qwen2.5-3b --methods baseline repetition cross_lingual_t1 cross_lingual_t2 --out {OUT}

In [ ]:
# CELL 12 — Punjabi (Shahmukhi)
!python /kaggle/working/prompt_repitition_research/run_all_languages.py --language pa --models qwen2.5-3b --methods baseline repetition cross_lingual_t1 cross_lingual_t2 --out {OUT}

In [ ]:
# CELL 13 — Pashto
!python /kaggle/working/prompt_repitition_research/run_all_languages.py --language ps --models qwen2.5-3b --methods baseline repetition cross_lingual_t1 cross_lingual_t2 --out {OUT}

In [ ]:
# CELL 14 — Balochi (5/7 tasks: OpenBookQA & CommonSenseQA translation blocked)
!python /kaggle/working/prompt_repitition_research/run_all_languages.py --language bal --models qwen2.5-3b --methods baseline repetition cross_lingual_t1 cross_lingual_t2 --out {OUT}

## Resume after a disconnect

Rerun the exact same command — the JSON is resume-safe and continues from the last saved item:

In [ ]:
# CELL 15 — Resume example (rerun interrupted language)
!python /kaggle/working/prompt_repitition_research/run_all_languages.py --language ar --models qwen2.5-3b --methods baseline repetition --out {OUT}

## Other models (optional)

Open-access: `qwen2.5-1.5b`, `qwen2.5-3b`, `qwen2.5-7b`, `mistral-7b-v0.3`, `gemma2-2b`

Gated (require HuggingFace login first): `llama3.2-1b`, `llama3.2-3b`, `llama3.1-8b`

In [ ]:
# CELL 16 — (Optional) HF login, required ONLY for Llama models
from huggingface_hub import login
login(token='hf_YOUR_TOKEN_HERE')  # get token from https://huggingface.co/settings/tokens

# Example: run Arabic with Llama 3.1 8B
!python /kaggle/working/prompt_repitition_research/run_all_languages.py --language ar --models llama3.1-8b --methods baseline repetition --out {OUT}

In [ ]:
# CELL 17 — (Optional) Zip results for download
os.system(f'cd {OUT} && zip -r rtl_results.zip .')
print('Zip at /kaggle/working/rtl_results/rtl_results.zip')
print('Download via the Output sidebar (Data section), or Save Version to archive it')

## Notes

- **Persistence:** everything in `/kaggle/working` is saved with the notebook when you click **Save Version** — results survive disconnects and sessions.
- **Free quota:** 30 GPU-hours/week (T4x2). Reset weekly.
- **Regime (per paper):** temperature = 0, max_tokens = 100, strict non-reasoning system prompts.
- **Runtime estimates (T4, qwen2.5-3b, all 4 methods):** Arabic ~2-3 hrs, Persian ~4-5 hrs, others ~1-2 hrs each.
- Model cache (`~/.cache/huggingface`) is wiped when the session is recycled — models re-download on first run.